## Flow control test case

In [1]:
# IMPORT UTILITIES

import random
import numpy as np
import torch

random.seed(0)
np.random.seed(0)
torch.manual_seed(0);

In [2]:
# PROBLEM DEFINITION

from utils.flowcontrol import FlowControl

flowcontrol = FlowControl(mu = 0.01, v_in_mod = 10.0, beta = 1.0, beta_g = 0.2, burnin = 5, dt = 0.1, T = 5.0)

In [ ]:
# SOLUTION EXAMPLE

alpha_in = 0.5

[vt, pt, J] = flowcontrol.solve_NS(alpha_in)

In [ ]:
# VELOCITY VISUALIZATION

from utils.postprocessing import trajectory

trajectory(vt,
           flowcontrol.plot_velocity,
           title = 'Velocity',
           times = flowcontrol.times)

## Data loading

In [ ]:
# DATA GENERATION (SKIP THIS CELL IF DATA ALREADY AVAILABLE)

ntrajectories = 100
parameters_ranges = [(-1.0, 1.0)] # Scenario parameter: angle of attack
nparameters = len(parameters_ranges)

V = np.empty((ntrajectories, flowcontrol.burnin + flowcontrol.ntimesteps, flowcontrol.nvelocity))
U = np.empty((ntrajectories, flowcontrol.ntimesteps - 1, flowcontrol.ncontrol))
MU = np.empty((ntrajectories, flowcontrol.burnin + flowcontrol.ntimesteps, nparameters))

for i in range(ntrajectories):

    print(f"Generating snapshots n.{i}...")
    
    # generate random parameters
    parameter = (parameters_ranges[0][1] - parameters_ranges[0][0]) * np.random.rand() + parameters_ranges[0][0]    
    
    # solve optimal control problem
    vt_opt, pt_opt, ut_opt, J_opt = flowcontrol.solve_OCP(parameter)
    
    # store results
    V[i] = vt_opt
    U[i] = ut_opt
    MU[i] = parameter
    
    np.savez("data/FlowControl/flowcontrol.npz", velocity = V, control = U, parameters = MU)

print("Snapshots generated!")

In [3]:
# LOAD SNAPSHOTS MATRICES

from utils.datamanager import DataManager

parameters_ranges = [(-1.0, 1.0)] # Parameter: angle of attack
nparameters = len(parameters_ranges)

datamanager = DataManager(np.load("data/FlowControl/flowcontrol.npz"),
                          dims = {'velocity': 2, 'control': 2},
                          train_ratio = 0.8,
                          valid_ratio = 0.1,
                          test_ratio = 0.1)

In [ ]:
# VELOCITY VISUALIZATION

which_trajectory = 0

trajectory(datamanager.data["velocity"][which_trajectory],
           flowcontrol.plot_velocity,
           title = 'Velocity',
           times = flowcontrol.times)

In [ ]:
# CONTROL VISUALIZATION

which_trajectory = 0

trajectory(datamanager.data["control"][which_trajectory],
           flowcontrol.plot_control,
           title = 'Control',
           times = flowcontrol.times[flowcontrol.burnin:])

In [ ]:
# CONTROL & VELOCITY VISUALIZATION

which_trajectory = 0

trajectory(datamanager.data["velocity"][which_trajectory, :-1],
           flowcontrol.plot_velocity_with_control,
           ut = np.concatenate((np.zeros((flowcontrol.burnin, flowcontrol.ncontrol)), datamanager.data["control"][which_trajectory]), axis = 0),
           title = 'Velocity with control',
           times = flowcontrol.times)

In [4]:
# TRAIN-VALIDATION-TEST SPLITTING

datamanager.prepare()

## Data compression

In [ ]:
# PROPER ORTHOGONAL DECOMPOSITION (POD)

kvelocity = 200
kcontrol = 20
datamanager.POD(ranks = {'velocity': kvelocity, 'control': kcontrol},
                starting_times = {'velocity': flowcontrol.burnin+1, 'control': 0})

## SHRED-ROM

In [ ]:
# EXTRACT SENSORS DATA (SKIP THIS CELL IF DATA ALREADY AVAILABLE)

nsensors = 3
nvelocity = datamanager.data["velocity"].shape[2]
sensors = np.random.choice(nvelocity // 2, size = nsensors, replace = False)
sensors_coordinates = torch.from_numpy(flowcontrol.Vh.sub(0).collapse().tabulate_dof_coordinates()[sensors]).reshape(-1)

sensors_data_train = datamanager.data["velocity"][:,:,0 : nvelocity : 2][datamanager.train,flowcontrol.burnin+1:,:][:,:,sensors]
sensors_data_valid = datamanager.data["velocity"][:,:,0 : nvelocity : 2][datamanager.valid,flowcontrol.burnin+1:,:][:,:,sensors]
sensors_data_test = datamanager.data["velocity"][:,:,0 : nvelocity : 2][datamanager.test,flowcontrol.burnin+1:,:][:,:,sensors]

In [6]:
# LOAD SENSORS DATA

nsensors = 3
nvelocity = datamanager.data["velocity"].shape[2]
sensors = torch.load('data/FlowControl/sensors.pt', weights_only = False)
sensors_coordinates = torch.from_numpy(flowcontrol.Vh.sub(0).collapse().tabulate_dof_coordinates()[sensors]).reshape(-1)

sensors_data_train = datamanager.data["velocity"][:,:,0 : nvelocity : 2][datamanager.train,flowcontrol.burnin+1:,:][:,:,sensors]
sensors_data_valid = datamanager.data["velocity"][:,:,0 : nvelocity : 2][datamanager.valid,flowcontrol.burnin+1:,:][:,:,sensors]
sensors_data_test = datamanager.data["velocity"][:,:,0 : nvelocity : 2][datamanager.test,flowcontrol.burnin+1:,:][:,:,sensors]

In [7]:
# BUILD TRAIN, VALIDATION AND TEST DATASETS WITH PADDING

from utils.datamanager import Padding, TimeSeriesDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lag = 20

train_data_in = Padding(torch.from_numpy(sensors_data_train).float(), lag).to(device)
valid_data_in = Padding(torch.from_numpy(sensors_data_valid).float(), lag).to(device)
test_data_in = Padding(torch.from_numpy(sensors_data_test).float(), lag).to(device)

train_data_out = Padding(torch.cat((torch.from_numpy(datamanager.data_POD["control"][datamanager.train,:,:]).float(),
                                    torch.from_numpy(datamanager.data_POD["velocity"][datamanager.train,:,:]).float()), 2),
                        1).squeeze(1).to(device)
valid_data_out = Padding(torch.cat((torch.from_numpy(datamanager.data_POD["control"][datamanager.valid,:,:]).float(),
                                    torch.from_numpy(datamanager.data_POD["velocity"][datamanager.valid,:,:]).float()), 2),
                        1).squeeze(1).to(device)
test_data_out = Padding(torch.cat((torch.from_numpy(datamanager.data_POD["control"][datamanager.test,:,:]).float(),
                                   torch.from_numpy(datamanager.data_POD["velocity"][datamanager.test,:,:]).float()), 2),
                        1).squeeze(1).to(device)

train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
test_dataset = TimeSeriesDataset(test_data_in, test_data_out)

In [ ]:
# SHRED TRAINING (SKIP THIS CELL IF MODEL ALREADY AVAILABLE)

from utils.models import SHRED, fit

nlatent = 64
shred = SHRED(nsensors, kcontrol + kvelocity, hidden_size = nlatent, hidden_layers = 2, decoder_sizes = [350, 400], dropout = 0.1).to(device)
train_errors, valid_errors = fit(shred, train_dataset, valid_dataset, batch_size = 64, epochs = 500, lr = 1e-3, verbose = True, patience = 100)
train_errors, valid_errors = fit(shred, train_dataset, valid_dataset, batch_size = 64, epochs = 500, lr = 1e-4, verbose = True, patience = 100)

In [9]:
# LOAD PRE-TRAINED SHRED MODEL

from utils.models import SHRED

nlatent = 64
shred = SHRED(nsensors, kcontrol + kvelocity, hidden_size = nlatent, hidden_layers = 2, decoder_sizes = [350, 400], dropout = 0.1).to(device)
shred.load_state_dict(torch.load('data/FlowControl/shred.pt', weights_only = False, map_location = torch.device('cpu')));

In [ ]:
# SHRED ERRORS ON TEST DATA

from utils.postprocessing import mre_numpy, num2p

shred.freeze()

shred_pred = shred(test_data_in).cpu().numpy()
shred_pred = shred_pred.reshape(datamanager.ntest, -1, kcontrol + kvelocity)

data_test_pred = datamanager.decode(data_POD = {'control': shred_pred[:,:,:kcontrol], 'velocity': shred_pred[:,:,kcontrol:]})

print(f"MRE on control: {num2p(mre_numpy(datamanager.data['control'][datamanager.test], data_test_pred['control']))}")
print(f"MRE on velocity: {num2p(mre_numpy(datamanager.data['velocity'][datamanager.test, flowcontrol.burnin+1:], data_test_pred['velocity']))}")

In [ ]:
# GROUND TRUTH vs SHRED-ROM PREDICTION

from utils.postprocessing import trajectories

which_trajectory = 0

trajectories([datamanager.data["velocity"][datamanager.test, flowcontrol.burnin+1:][which_trajectory],
              data_test_pred['velocity'][which_trajectory],
              np.sqrt((datamanager.data["velocity"][datamanager.test, flowcontrol.burnin+1:][which_trajectory] - data_test_pred['velocity'][which_trajectory])**2)],
              flowcontrol.plot_velocity,
              titles = ["Ground truth", "SHRED-ROM prediction", "Prediction error"],
              times = flowcontrol.times[flowcontrol.burnin:],
              figsize = (15, 5))

In [ ]:
# GROUND TRUTH vs SHRED-ROM PREDICTION

which_trajectory = 0

trajectories([datamanager.data["control"][datamanager.test][which_trajectory],
              data_test_pred['control'][which_trajectory],
              np.sqrt((datamanager.data["control"][datamanager.test][which_trajectory] - data_test_pred['control'][which_trajectory])**2)],
              flowcontrol.plot_control,
              titles = ["Ground truth", "SHRED-ROM prediction", "Prediction error"],
              times = flowcontrol.times[flowcontrol.burnin:],
              figsize = (15, 5))

In [ ]:
# GROUND TRUTH vs SHRED-ROM PREDICTION

which_trajectory = 0

trajectories([datamanager.data["velocity"][datamanager.test, flowcontrol.burnin+1:][which_trajectory],
              data_test_pred['velocity'][which_trajectory],
              np.sqrt((datamanager.data["velocity"][datamanager.test, flowcontrol.burnin+1:][which_trajectory] - data_test_pred['velocity'][which_trajectory])**2)],
              flowcontrol.plot_velocity_with_control,
              uts = [datamanager.data["control"][datamanager.test][which_trajectory],
                     data_test_pred['control'][which_trajectory],
                     np.sqrt((datamanager.data["control"][datamanager.test][which_trajectory] - data_test_pred['control'][which_trajectory])**2)],
              titles = ["Ground truth", "SHRED-ROM prediction", "Prediction error"],
              times = flowcontrol.times[flowcontrol.burnin:],
              figsize = (15, 5))

## Latent sensor forecaster

In [11]:
# LOAD SENSORS DATA

nsensors = 3
nvelocity = datamanager.data["velocity"].shape[2]
sensors = torch.load('data/FlowControl/sensors.pt', weights_only = False)
sensors_coordinates = torch.from_numpy(flowcontrol.Vh.sub(0).collapse().tabulate_dof_coordinates()[sensors]).reshape(-1)

sensors_data_train = datamanager.data["velocity"][:,:,0 : nvelocity : 2][datamanager.train,flowcontrol.burnin+1:,:][:,:,sensors]
sensors_data_valid = datamanager.data["velocity"][:,:,0 : nvelocity : 2][datamanager.valid,flowcontrol.burnin+1:,:][:,:,sensors]
sensors_data_test = datamanager.data["velocity"][:,:,0 : nvelocity : 2][datamanager.test,flowcontrol.burnin+1:,:][:,:,sensors]

In [12]:
# LOAD PRE-TRAINED SHRED MODEL

from utils.models import SHRED

nlatent = 64
shred = SHRED(nsensors, kcontrol + kvelocity, hidden_size = nlatent, hidden_layers = 2, decoder_sizes = [350, 400], dropout = 0.1).to(device)
shred.load_state_dict(torch.load('data/FlowControl/shred.pt', weights_only = False, map_location = torch.device('cpu')));
shred.freeze()

In [13]:
# BUILD TRAIN, VALIDATION AND TEST DATASETS WITH PADDING

from utils.datamanager import Padding, TimeSeriesDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lag = 20

z_train = shred.sensor2latent(train_data_in).reshape(datamanager.ntrain, -1, nlatent)[:,:-1,:]
z_valid = shred.sensor2latent(valid_data_in).reshape(datamanager.nvalid, -1, nlatent)[:,:-1,:]
z_test = shred.sensor2latent(test_data_in).reshape(datamanager.ntest, -1, nlatent)[:,:-1,:]

train_data_in_forecaster = Padding(torch.cat((torch.from_numpy(sensors_data_train[:,:-1,:]), z_train), 2), lag).to(device)
valid_data_in_forecaster = Padding(torch.cat((torch.from_numpy(sensors_data_valid[:,:-1,:]), z_valid), 2), lag).to(device)
test_data_in_forecaster = Padding(torch.cat((torch.from_numpy(sensors_data_test[:,:-1,:]), z_test), 2), lag).to(device)

train_data_out_forecaster = Padding(torch.from_numpy(sensors_data_train[:,1:,:]), 1).squeeze(1).to(device)
valid_data_out_forecaster = Padding(torch.from_numpy(sensors_data_valid[:,1:,:]), 1).squeeze(1).to(device)
test_data_out_forecaster = Padding(torch.from_numpy(sensors_data_test[:,1:,:]), 1).squeeze(1).to(device)

train_dataset_forecaster = TimeSeriesDataset(train_data_in_forecaster, train_data_out_forecaster)
valid_dataset_forecaster = TimeSeriesDataset(valid_data_in_forecaster, valid_data_out_forecaster)
test_dataset_forecaster = TimeSeriesDataset(test_data_in_forecaster, test_data_out_forecaster)

In [ ]:
# LATENT SENSOR FORECASTER TRAINING

from utils.models import LatentSensorForecaster, fit

latentsensorforecaster = LatentSensorForecaster(nsensors + nlatent, nsensors, hidden_size = 64, hidden_layers = 2, dropout = 0.1).to(device)
train_errors, valid_errors = fit(latentsensorforecaster, train_dataset_forecaster, valid_dataset_forecaster, batch_size = 64, epochs = 500, lr = 1e-3, verbose = True, patience = 100)
train_errors, valid_errors = fit(latentsensorforecaster, train_dataset_forecaster, valid_dataset_forecaster, batch_size = 64, epochs = 500, lr = 1e-4, verbose = True, patience = 100)

In [15]:
# LOAD PRE-TRAINED LATENT SENSOR FORECASTER

from utils.models import LatentSensorForecaster

latentsensorforecaster = LatentSensorForecaster(nsensors + nlatent, nsensors, hidden_size = 64, hidden_layers = 2, dropout = 0.1).to(device)
latentsensorforecaster.load_state_dict(torch.load('data/FlowControl/latentsensorforecaster.pt', weights_only = False, map_location = torch.device('cpu')));

In [ ]:
# LATENT SENSOR FORECASTER ERRORS ON TEST DATA

from utils.postprocessing import mre_numpy, num2p

latentsensorforecaster.freeze()

sensor_test_hat = latentsensorforecaster(test_data_in_forecaster).cpu().numpy().reshape(datamanager.ntest, -1, nsensors)

print(f"Mean relative latent sensor forecaster prediction error: {num2p(mre_numpy(sensors_data_test[:,1:,:], sensor_test_hat))}")

## SHRED-ROM into play

In [ ]:
# FLOW CONTROL FOR A TEST PARAMETER

from fenics import Function
import matplotlib.pyplot as plt
from IPython.display import clear_output as clc

alpha_in_test = 0.25
sensors_test = torch.zeros(flowcontrol.ntimesteps + lag - 1, nsensors)

# Burn-in
Js = []
for j in range(flowcontrol.burnin):
    u = Function(flowcontrol.Vh)
    if j == 0:
        [vt, pt, J] = flowcontrol.solve_NS(alpha_in_test, ut = [u], ntimesteps = 2)
    else:
        [vt, pt, J] = flowcontrol.solve_NS(alpha_in_test, v0_val = vt[-1], p0_val = pt[-1], ut = [u], ntimesteps = 2)
    Js.append(J)

for j in range(flowcontrol.ntimesteps):
   
    # Extract sensor values
    sensors_test[j+lag-1] = torch.from_numpy(vt[-1,::2][sensors])

    # Predict optimal control
    shred_pred = shred(sensors_test[j:j+lag,:].unsqueeze(0)).cpu().numpy()
    shred_pred = shred_pred.reshape(1, 1, kcontrol + kvelocity)
    data_test_pred = datamanager.decode(data_POD = {'control': shred_pred[:,:,:kcontrol], 'velocity': shred_pred[:,:,kcontrol:]})
        
    # Apply optimal control
    u_val = np.zeros(flowcontrol.nvelocity)
    u_val[flowcontrol.idx_control] = data_test_pred["control"][0,0]
    u = Function(flowcontrol.Vh)
    u.vector()[:] = u_val
    [vt, pt, J] = flowcontrol.solve_NS(alpha_in_test, v0_val = vt[-1], p0_val = pt[-1], ut = [u], ntimesteps = 2)

    # Store result
    Js.append(J)

clc()
plt.plot(Js, 'magenta', linewidth = 2)
plt.axvline(x = flowcontrol.burnin, linestyle='--')
plt.xlabel("Time")
plt.ylabel("Loss function");

In [ ]:
# FLOW CONTROL FOR A TEST PARAMETER WITH MISSING SENSOR DATA

from fenics import Function
import matplotlib.pyplot as plt
from IPython.display import clear_output as clc

alpha_in_test = 0.25
sensors_test = torch.zeros(flowcontrol.ntimesteps + lag - 1, nsensors)
z_test = torch.zeros(flowcontrol.ntimesteps + lag - 1, nlatent)

# Burn-in
Js = []
for j in range(flowcontrol.burnin):
    u = Function(flowcontrol.Vh)
    if j == 0:
        [vt, pt, J] = flowcontrol.solve_NS(alpha_in_test, ut = [u], ntimesteps = 2)
    else:
        [vt, pt, J] = flowcontrol.solve_NS(alpha_in_test, v0_val = vt[-1], p0_val = pt[-1], ut = [u], ntimesteps = 2)
    Js.append(J)

for j in range(flowcontrol.ntimesteps):
   
    # Extract sensor values
    if j < 25: # Sensors ON
        sensors_test[j+lag-1] = torch.from_numpy(vt[-1,::2][sensors])
    else: # Sensors OFF
        sensors_test[j+lag-1] = latentsensorforecaster(torch.cat((sensors_test[j-1:j+lag-1,:].unsqueeze(0), z_test[j-1:j+lag-1,:].unsqueeze(0)),2).cpu())[0]
    z_test[j+lag-1] = shred.sensor2latent(sensors_test[j:j+lag,:].unsqueeze(0)).cpu()

    # Predict optimal control
    shred_pred = shred(sensors_test[j:j+lag,:].unsqueeze(0)).cpu().numpy()
    shred_pred = shred_pred.reshape(1, 1, kcontrol + kvelocity)
    data_test_pred = datamanager.decode(data_POD = {'control': shred_pred[:,:,:kcontrol], 'velocity': shred_pred[:,:,kcontrol:]})
        
    # Apply optimal control
    u_val = np.zeros(flowcontrol.nvelocity)
    u_val[flowcontrol.idx_control] = data_test_pred["control"][0,0]
    u = Function(flowcontrol.Vh)
    u.vector()[:] = u_val
    [vt, pt, J] = flowcontrol.solve_NS(alpha_in_test, v0_val = vt[-1], p0_val = pt[-1], ut = [u], ntimesteps = 2)

    # Store result
    Js.append(J)

clc()
plt.plot(Js, 'magenta', linewidth = 2)
plt.axvline(x = flowcontrol.burnin, linestyle='--')
plt.xlabel("Time")
plt.ylabel("Loss function");